## 06 Incremental Model Querying 編


In [ ]:
!pip install langchain==0.3.0 langchain-openai==0.2.0 langgraph==0.2.22 langchain-community pydantic==2.10.6 dotenv faiss-cpu

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY") # LangSmith 連携用
os.environ["LANGCHAIN_TRACING_V2"] = "true" # トレース有効化
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


In [3]:
# ===============================
# セル1: 必要ライブラリのインポート
# ===============================

from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage


In [4]:
# ===============================
# セル2: モデルの初期化
# ===============================
# ここでは gpt-4o-mini を例に使用
# （環境によって "gpt-4o-mini" → "gpt-4" などに変更可能）

llm = ChatOpenAI(model="gpt-4o-mini")


C:\Users\user\AppData\Local\Temp\ipykernel_27080\1720745342.py:7: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-4o-mini")


In [5]:
# ===============================
# セル3: Incremental Query の準備
# ===============================
# ミャクミャケがもやもやして投げたいテーマを「小分け」に分解する

questions = [
    "行政手続きにおける『紙の手続き』のメリットを3つ挙げてください。",
    "行政手続きにおける『デジタル手続き』のメリットを3つ挙げてください。",
    "紙とデジタルの両方を共存させる方法を提案してください。"
]


In [6]:
# ===============================
# セル4: 段階的に問い合わせる
# ===============================
# 1問ずつモデルに投げて、回答を蓄積していく

answers = []

for idx, q in enumerate(questions, 1):
    response = llm([HumanMessage(content=q)])
    print(f"Q{idx}: {q}")
    print("A:", response.content, "\n")
    answers.append(response.content)


C:\Users\user\AppData\Local\Temp\ipykernel_27080\2002729446.py:9: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm([HumanMessage(content=q)])


Q1: 行政手続きにおける『紙の手続き』のメリットを3つ挙げてください。
A: 行政手続きにおける『紙の手続き』のメリットを以下の3つ挙げます。

1. **物理的な証拠の保持**: 紙の手続きでは、書類として物理的に存在するため、提出した内容や日付、受け取りの証拠を容易に保管できます。これにより、後日トラブルが発生した場合でも、証拠として利用できる点がメリットです。

2. **視覚的な確認が容易**: 紙の書類は、視覚的に確認しやすく、情報の整理や記載内容の見直しが直感的に行いやすいです。また、手続きの進捗状況や関連書類を一目で把握できるため、管理がしやすいという利点があります。

3. **デジタルデバイドの解消**: 高齢者やITに不慣れな人々にとって、紙の手続きは使いやすい場合があります。デジタル機器やインターネット環境にアクセスできない人々に対しても、手続きの機会を平等に提供できるため、行政サービスの普及に寄与します。 

Q2: 行政手続きにおける『デジタル手続き』のメリットを3つ挙げてください。
A: 行政手続きにおける『デジタル手続き』のメリットを以下の3つ挙げます。

1. **効率性の向上**: デジタル手続きを導入することで、書類の作成や提出、処理にかかる時間を大幅に短縮できます。オンライン申請システムを利用することで、窓口での待ち時間を減らし、迅速な対応が可能になります。また、データの自動処理により、人的ミスを減少させることも期待できます。

2. **アクセスの向上**: デジタル手続きは、場所や時間に制約されずに利用できるため、より多くの市民が行政サービスにアクセスしやすくなります。特に、遠隔地に住む人や、移動が困難な高齢者や障害者にとっては、大きな利点です。

3. **コスト削減**: 紙の使用を減らすことで印刷や郵送にかかるコストを削減できるほか、行政機関の運営コストも低減されます。また、デジタル化により業務の効率化が進むことで、人的リソースを他の重要な業務に振り分けることができ、全体的なコストの最適化が図れます。

これらのメリットにより、デジタル手続きは行政サービスの質を向上させる重要な手段となります。 

Q3: 紙とデジタルの両方を共存させる方法を提案してください。
A: 紙とデジタルの両方を効果的に共存させるための方法は

In [7]:
# ===============================
# セル5: 最後にまとめを依頼
# ===============================
# これまでの回答を統合して、ハイブリッド戦略を要約

summary_prompt = "以下の回答を統合して、行政手続きの簡素化戦略を提案してください:\n\n"
summary_prompt += "\n".join(answers)

final_summary = llm([HumanMessage(content=summary_prompt)])
print("=== 最終まとめ ===")
print(final_summary.content)


=== 最終まとめ ===
行政手続きの簡素化戦略として、以下の提案をまとめました。この戦略は、紙とデジタルの手続きを効果的に融合し、利便性を向上させることを目指します。

### 1. ハイブリッド型手続きの導入
紙とデジタルの両方のメリットを活かし、ハイブリッド型の手続きを導入します。具体的には、重要な書類はデジタルで提出可能にしつつ、必要に応じて紙の書類を保持することを奨励します。これにより、物理的な証拠も確保されつつ、デジタル化による効率性も享受できます。

### 2. デジタル化の推進
すべての手続きが可能な限りデジタル化されるよう、オンライン申請システムを整備します。これにより、申請から処理までの時間を短縮し、コスト削減も図ることができます。また、デジタルデバイスを活用して、手続きに必要な情報をスマートフォンやタブレットで簡単に確認できるようにします。

### 3. アクセスの向上
特にITに不慣れな高齢者や障害者向けに、紙の手続きを選択できるオプションを維持しますが、同時にデジタルツールの使い方に関する教育やトレーニングを実施し、デジタルデバイドの解消を図ります。

### 4. 情報整理と管理の効率化
デジタルツールを用いて、紙の書類をスキャンしデジタルデータとして保存することで、物理的なスペースを節約しつつ、情報の整理を容易にする施策を推進します。また、定期的なデータのバックアップを行い、デジタル情報の損失を防ぎます。

### 5. ペーパーレス化の促進
可能な限りペーパーレス化を進め、紙の使用を減少させます。デジタル化できる書類は積極的にデジタル化し、紙が必要な場合は質の良い紙を使用するなど、環境への配慮も含めた施策を実施します。

### 6. 情報の可視化とタスク管理
プロジェクト管理やタスク管理においては、デジタルツールを活用しつつ、重要な情報は紙のカレンダーやホワイトボードに記入して可視化することで、両方の利点を生かします。

### 7. 定期的な見直しと改善
行政手続きの実施後は、定期的にプロセスの見直しを行い、利用者からのフィードバックを基に改善を図ります。このことで、手続きの利便性を継続的に向上させることができます。

この戦略を通じて、行政手続きがより効率的かつ市民に優しいものとなり、全体的な行政サービスの質向上に寄与する